In [1]:
import numpy as np
from scipy.stats import norm
from scipy.stats import skewnorm
import statsmodels.api as sm

from tqdm.notebook import tqdm
import pickle
import pandas as pd
import os
import lightning as L
from lightning.pytorch.loggers import TensorBoardLogger

import sys 
sys.path.insert(0, '../src/')

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)

In [2]:
# VIDS here
import vids_segm_cld.vids_segm as vids
import datasets.echonet

In [3]:
path_to_data_train = '/home/paul/Downloads/echonet_dynamic_preprocessed/TRAIN/'
path_to_data_val = '/home/paul/Downloads/echonet_dynamic_preprocessed/VAL/'

In [4]:
loader_train = datasets.echonet.load_data_into_loader(16, path_to_data_train, shuffle=True)
loader_val = datasets.echonet.load_data_into_loader(16, path_to_data_val, shuffle=False)

Number of samples: 200
Number of samples: 200


In [5]:
n_channels = 1       # grayscale
num_classes = 2       # binary segmentation
embedding_dim = 32    # per-pixel embedding dimension
H, W = 112, 112      # image size

In [6]:
# --- Step 1: Create and pre-train embedding network ---
print("Step 1: Pre-training embedding network...")
embedding_net = vids.UNetDenseEmbedding(
    n_channels=n_channels, 
    embedding_dim=embedding_dim, 
    bilinear=False
)

Step 1: Pre-training embedding network...


In [7]:
embedding_net, pretrained_model = vids.pretrain_segmentation_embedding(
    embedding_net=embedding_net,
    loader=loader_train,
    num_classes=num_classes,
    epochs=3,
    lr=1e-4
)

Epoch: 0


  Pre-train Epoch 1/3, Loss: 0.713739, Dice: 0.3905
Epoch: 1
  Pre-train Epoch 2/3, Loss: 0.549916, Dice: 0.4843
Epoch: 2
  Pre-train Epoch 3/3, Loss: 0.468586, Dice: 0.5322


In [8]:
 # --- Step 2: Create VIDS model ---
print("\nStep 2: Creating VIDS model...")

# For segmentation with small theta, we need small inference network
theta_dim = embedding_dim * num_classes + num_classes  # 32*2 + 2 = 66
inference_hidden = [256, 128, 64]  # Smaller than default

vids_model = vids.VIDS(
    embedding_net=embedding_net,
    embedding_dim=embedding_dim,
    output_dim=num_classes,
    task="segmentation",
    inference_hidden_dims=inference_hidden,
    kl_weight=0.001,
    variance_penalty=0.001,
    num_classes=num_classes,
)

print(f"  Prediction head params (θ): {vids_model.prediction_head.num_params}")
print(f"  Inference network params: {sum(p.numel() for p in vids_model.inference_net.parameters())}")


Step 2: Creating VIDS model...
  Prediction head params (θ): 66
  Inference network params: 66372


In [11]:
# --- Step 3: Train VIDS ---
print("\nStep 3: Training VIDS inference network...")
losses = vids.train_vids(
    model=vids_model,
    train_loader=loader_train,
    num_environments=30,
    env_train_size=30,
    env_test_size=30,
    num_epochs=50,
    lr=1e-3,
    verbose=True,
)


Step 3: Training VIDS inference network...
  Epoch 1/10 [Batch 10], Loss: 66211088.0000
  Epoch 2/10 [Batch 8], Loss: 4533776.5000


KeyboardInterrupt: 